# RAG system for PDF's using Ollama service with LlamaIndex

This notebook is to build a system that leverages the power of LlamaIndex to retrieve and integrate data into responses, ensuring that the content generated is always accurate and up-to-date.
- Use LlamaIndex to construct a RAG application that retrieve information from documents.
- Load, index, and retrieve data efficiently to ensure your RAG application accesses the most relevant information.
- Enhance querying techniques with LlamaIndex for precise and context-aware responses.

## Setup

In [1]:
%%time
import warnings
warnings.filterwarnings("ignore")

!python3 -m pip install --upgrade pip

CPU times: user 8.38 ms, sys: 10 ms, total: 18.4 ms
Wall time: 1.22 s


### OSS libraries install

In [2]:
%pip install llama-index llama-index-core llama-index-embeddings-huggingface llama-index-llms-ollama ollama ipython-autotime --use-deprecated=legacy-resolver


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
%load_ext autotime

time: 64.9 μs (started: 2025-07-31 11:32:47 -07:00)


### My variables

In [4]:
my_model_ollama = "llama3.2"

time: 162 μs (started: 2025-07-31 11:32:47 -07:00)


## Loading

In [5]:
import os

print(f"Notebook path: {os.getcwd()}")

Notebook path: /Users/krishnam/Documents/GitHub-KM/examples.ipynb/LLM/RAG
time: 441 μs (started: 2025-07-31 11:32:47 -07:00)


In [6]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(input_files=["./docs/lora_paper.pdf"]).load_data()
documents

[Document(id_='84971e69-84e1-4a54-bb16-e37530fe817e', embedding=None, metadata={'page_label': '1', 'file_name': 'lora_paper.pdf', 'file_path': 'docs/lora_paper.pdf', 'file_type': 'application/pdf', 'file_size': 1609513, 'creation_date': '2025-07-08', 'last_modified_date': '2025-07-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='LORA: L OW-RANK ADAPTATION OF LARGE LAN-\nGUAGE MODELS\nEdward Hu∗ Yelong Shen∗ Phillip Wallis Zeyuan Allen-Zhu\nYuanzhi Li Shean Wang Lu Wang Weizhu Chen\nMicrosoft Corporation\n{edwardhu, yeshe, phwallis, zeyuana,\nyuanzhil, swang, luw, wzchen}@microsoft.com\nyuanzhil@andrew.cmu.edu\n(Version 2)\nABSTRACT\nA

time: 4.03 s (started: 2025-07-31 11:32:47 -07:00)


### Splitting

In [7]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=500)
splitter

SentenceSplitter(include_metadata=True, include_prev_next_rel=True, callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x14712f110>, id_func=<function default_id_func at 0x121d8a7a0>, chunk_size=500, chunk_overlap=200, separator=' ', paragraph_separator='\n\n\n', secondary_chunking_regex='[^,.;。？！]+[,.;。？！]?|[,.;。？！]')

time: 105 ms (started: 2025-07-31 11:32:51 -07:00)


In [8]:
nodes = splitter.get_nodes_from_documents(documents)
nodes

[TextNode(id_='8a61ae4b-4336-45a5-985a-1041439294f7', embedding=None, metadata={'page_label': '1', 'file_name': 'lora_paper.pdf', 'file_path': 'docs/lora_paper.pdf', 'file_type': 'application/pdf', 'file_size': 1609513, 'creation_date': '2025-07-08', 'last_modified_date': '2025-07-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='84971e69-84e1-4a54-bb16-e37530fe817e', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'page_label': '1', 'file_name': 'lora_paper.pdf', 'file_path': 'docs/lora_paper.pdf', 'file_type': 'application/pdf', 'file_size': 1609513, 'creation_date': '2025-07-08', 'last_modified_date': '2025-07-07'}, hash='bb26fe3db6a252cd94e16d5df293ce17d26fbeed41e1062a9b02fb13eb6892b5'), <NodeRelati

time: 1.53 s (started: 2025-07-31 11:32:51 -07:00)


In [9]:
len(nodes)

90

time: 11.1 ms (started: 2025-07-31 11:32:53 -07:00)


In [10]:
node_metadata = nodes[0].get_content(metadata_mode=True)
str(node_metadata)

'page_label: 1\nfile_name: lora_paper.pdf\nfile_path: docs/lora_paper.pdf\nfile_type: application/pdf\nfile_size: 1609513\ncreation_date: 2025-07-08\nlast_modified_date: 2025-07-07\n\nLORA: L OW-RANK ADAPTATION OF LARGE LAN-\nGUAGE MODELS\nEdward Hu∗ Yelong Shen∗ Phillip Wallis Zeyuan Allen-Zhu\nYuanzhi Li Shean Wang Lu Wang Weizhu Chen\nMicrosoft Corporation\n{edwardhu, yeshe, phwallis, zeyuana,\nyuanzhil, swang, luw, wzchen}@microsoft.com\nyuanzhil@andrew.cmu.edu\n(Version 2)\nABSTRACT\nAn important paradigm of natural language processing consists of large-scale pre-\ntraining on general domain data and adaptation to particular tasks or domains. As\nwe pre-train larger models, full ﬁne-tuning, which retrains all model parameters,\nbecomes less feasible. Using GPT-3 175B as an example – deploying indepen-\ndent instances of ﬁne-tuned models, each with 175B parameters, is prohibitively\nexpensive. We propose Low-Rank Adaptation, or LoRA, which freezes the pre-\ntrained model weights an

time: 3.41 ms (started: 2025-07-31 11:32:53 -07:00)


## Indexing
At a high level, Indexes are built from Documents. These Indexes are then used to construct Query Engines and Chat Engines, which power question-and-answer interactions and conversational experiences over your data.

In [11]:
from llama_index.embeddings.ollama import OllamaEmbedding
ollams_embedding = OllamaEmbedding(model_name=my_model_ollama, truncate_input_tokens=3)
ollams_embedding

OllamaEmbedding(model_name='llama3.2', embed_batch_size=10, callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x16d60b790>, num_workers=None, embeddings_cache=None, base_url='http://localhost:11434', ollama_additional_kwargs={})

time: 85.1 ms (started: 2025-07-31 11:32:53 -07:00)


## Storing

In [12]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex(
    nodes=nodes, 
    embed_model=ollams_embedding, 
    show_progress=True
)

vector_index

Generating embeddings:   0%|          | 0/90 [00:00<?, ?it/s]

time: 1min 16s (started: 2025-07-31 11:32:53 -07:00)


In [13]:
base_retriever = vector_index.as_retriever(similarity_top_k=3) # 3 for top 3 results

base_retriever

time: 1.07 ms (started: 2025-07-31 11:34:10 -07:00)


In [14]:
source_nodes = base_retriever.retrieve(my_model_ollama) # querying 

source_nodes

[NodeWithScore(node=TextNode(id_='ccb3c325-2147-4fb2-b0d7-ce6b5f8d3226', embedding=None, metadata={'page_label': '15', 'file_name': 'lora_paper.pdf', 'file_path': 'docs/lora_paper.pdf', 'file_type': 'application/pdf', 'file_size': 1609513, 'creation_date': '2025-07-08', 'last_modified_date': '2025-07-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='207bc5fc-e251-426c-9e9f-8c2f06521a0b', node_type='4', metadata={'page_label': '15', 'file_name': 'lora_paper.pdf', 'file_path': 'docs/lora_paper.pdf', 'file_type': 'application/pdf', 'file_size': 1609513, 'creation_date': '2025-07-08', 'last_modified_date': '2025-07-07'}, hash='4539a247973e18b2779a3149ba81f9ae3ed1eb06c9105b1f5a98bec52012fad3'), <NodeRelation

time: 149 ms (started: 2025-07-31 11:34:10 -07:00)


In [15]:
for node in source_nodes:
    # print(node.metadata)
    print(f"---------------------------------------------")
    print(f"Score: {node.score:.3f}")
    print(node.get_content())
    print(f"---------------------------------------------\n\n")

---------------------------------------------
Score: 0.514
Adapterdrop: On the efﬁciency of adapters in transformers, 2020.
Tara N Sainath, Brian Kingsbury, Vikas Sindhwani, Ebru Arisoy, and Bhuvana Ramabhadran. Low-
rank matrix factorization for deep neural network training with high-dimensional output targets.
In 2013 IEEE international conference on acoustics, speech and signal processing , pp. 6655–
6659. IEEE, 2013.
Mohammad Shoeybi, Mostofa Patwary, Raul Puri, Patrick LeGresley, Jared Casper, and Bryan
Catanzaro. Megatron-lm: Training multi-billion parameter language models using model par-
allelism, 2020.
Richard Socher, Alex Perelygin, Jean Wu, Jason Chuang, Christopher D. Manning, Andrew Ng,
and Christopher Potts. Recursive deep models for semantic compositionality over a sentiment
treebank. In Proceedings of the 2013 Conference on Empirical Methods in Natural Language
Processing, pp. 1631–1642, Seattle, Washington, USA, October 2013. Association for Computa-
tional Linguistic

## Querying

In [16]:
my_temperature = 0.1
my_max_new_tokens = 75
my_additional_params = {
    "decoding_method": "sample",
    "min_new_tokens": 1,
    "top_k": 50,
    "top_p": 1,
}

time: 181 μs (started: 2025-07-31 11:34:10 -07:00)


In [17]:
from llama_index.llms.ollama import Ollama

llm_client = Ollama(
    model=my_model_ollama,
    temperature=my_temperature,
    max_new_tokens=my_max_new_tokens,
    additional_params=my_additional_params,
)

llm_client

Ollama(callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x177e962d0>, system_prompt=None, messages_to_prompt=<function messages_to_prompt at 0x1271e98a0>, completion_to_prompt=<function default_completion_to_prompt at 0x12772c9a0>, output_parser=None, pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'>, query_wrapper_prompt=None, base_url='http://localhost:11434', model='llama3.2', temperature=0.1, context_window=-1, request_timeout=30.0, prompt_key='prompt', json_mode=False, additional_kwargs={}, is_function_calling_model=True, keep_alive=None, thinking=None)

time: 26 ms (started: 2025-07-31 11:34:10 -07:00)


In [18]:
response = llm_client.complete("What is a Generative AI?")

response

CompletionResponse(text='Generative AI refers to a type of artificial intelligence (AI) that can generate new, original content based on patterns and structures learned from existing data. This technology uses machine learning algorithms to create novel outputs, such as images, videos, text, music, or even entire articles.\n\nGenerative AI models are trained on large datasets, which enables them to learn the underlying patterns and relationships between different elements of the data. Once trained, these models can generate new content that is similar in style, tone, and structure to the training data.\n\nThere are several types of generative AI models, including:\n\n1. **Generative Adversarial Networks (GANs)**: These models use a competitive process between two neural networks to generate new content.\n2. **Variational Autoencoders (VAEs)**: These models learn to compress and reconstruct data, which can be used to generate new content.\n3. **Neural Style Transfer**: This technique al

time: 11.3 s (started: 2025-07-31 11:34:10 -07:00)


In [19]:
import pprint
pprint.pprint(response)

CompletionResponse(text='Generative AI refers to a type of artificial intelligence (AI) that can generate new, original content based on patterns and structures learned from existing data. This technology uses machine learning algorithms to create novel outputs, such as images, videos, text, music, or even entire articles.\n\nGenerative AI models are trained on large datasets, which enables them to learn the underlying patterns and relationships between different elements of the data. Once trained, these models can generate new content that is similar in style, tone, and structure to the training data.\n\nThere are several types of generative AI models, including:\n\n1. **Generative Adversarial Networks (GANs)**: These models use a competitive process between two neural networks to generate new content.\n2. **Variational Autoencoders (VAEs)**: These models learn to compress and reconstruct data, which can be used to generate new content.\n3. **Neural Style Transfer**: This technique al

In [20]:
query_engine = vector_index.as_query_engine(
  streaming=False, 
  similarity_top_k=7, 
  llm=llm_client)

query_engine

time: 11.4 ms (started: 2025-07-31 11:34:21 -07:00)


In [21]:
response = query_engine.query("What is the lora paper about?")
print(str(response))

The LORA (Low-Rank Adaptation) paper appears to be a research work focused on developing an efficient adaptation method for large pre-trained language models. The authors explore ways to improve the performance of these models without requiring extensive fine-tuning, which can be computationally expensive and require large amounts of data.

The paper discusses various approaches, including optimizing input word embeddings and using low-rank structures in deep learning. It also introduces a new method called LoRA, which uses a bottleneck structure to impose a low-rank constraint on the weight updates. The authors claim that this approach can improve the performance of pre-trained models while reducing the number of trainable parameters.

The paper presents experimental results on several benchmarks, including GLUE, SuperGLUE, and E2E NLG Challenge, where LoRA outperforms or is competitive with other state-of-the-art methods. Additionally, it explores the application of LoRA to GPT-2 mod

In [22]:
pprint.pprint(response)

Response(response='The LORA (Low-Rank Adaptation) paper appears to be a '
                  'research work focused on developing an efficient adaptation '
                  'method for large pre-trained language models. The authors '
                  'explore ways to improve the performance of these models '
                  'without requiring extensive fine-tuning, which can be '
                  'computationally expensive and require large amounts of '
                  'data.\n'
                  '\n'
                  'The paper discusses various approaches, including '
                  'optimizing input word embeddings and using low-rank '
                  'structures in deep learning. It also introduces a new '
                  'method called LoRA, which uses a bottleneck structure to '
                  'impose a low-rank constraint on the weight updates. The '
                  'authors claim that this approach can improve the '
                  'performance of pre-train

In [23]:
response = query_engine.query("List all the evaluation datasets that where used in the lora paper. Only consider the paper.")
print(str(response))

The evaluation datasets mentioned in the LoRA paper are:

1. WikiSQL
2. MultiNLI-matched (a variant of the MultiNLI dataset)
3. SAMSum
4. E2E NLG Challenge
5. DART
time: 9.91 s (started: 2025-07-31 11:34:36 -07:00)


In [24]:
pprint.pprint(response)

Response(response='The evaluation datasets mentioned in the LoRA paper are:\n'
                  '\n'
                  '1. WikiSQL\n'
                  '2. MultiNLI-matched (a variant of the MultiNLI dataset)\n'
                  '3. SAMSum\n'
                  '4. E2E NLG Challenge\n'
                  '5. DART',
         source_nodes=[NodeWithScore(node=TextNode(id_='690e1fea-e92d-4e3b-b173-d245d1161c65', embedding=None, metadata={'page_label': '20', 'file_name': 'lora_paper.pdf', 'file_path': 'docs/lora_paper.pdf', 'file_type': 'application/pdf', 'file_size': 1609513, 'creation_date': '2025-07-08', 'last_modified_date': '2025-07-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='e0d96e26-8e10-4e73-9d